# PS → FFN 软件替身 → PYNQ PL
本 notebook 默认只做 CPU 验证，不连接相机/串口/车辆、不修改 boot/CMA/设备树。PL 部分只有手动设置开关后才执行；需要已经验收的板卡和匹配 bit/hwh。主机输出不是 KV260 性能。
在 `pynq_ffn` 或其 `notebooks` 目录启动。Kernel 必须能 import NumPy/ONNX Runtime；启用 PL 时还需同一进程可用 PYNQ。

In [ ]:
from pathlib import Path
import sys
import numpy as np
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'ps' / 'runtime.py').is_file():
    raise RuntimeError('请从 pynq_ffn 或 notebooks 目录启动，或明确设置 ROOT')
sys.path.insert(0, str(ROOT))
from ps.runtime import Runtime
from ps.preprocess import read_records, preprocess_record
BUNDLE = ROOT / 'artifacts' / 'bundle_20261014'
if not BUNDLE.is_dir():
    raise RuntimeError('先用 tools/prepare_model.py 生成 bundle，并调整 BUNDLE')
print({'python': sys.executable, 'bundle': str(BUNDLE), 'hardware_execution': False})

## 1. 输入与整图 CPU 基线
优先使用随本次验证保存的单组三摄样本；不存在时使用固定种子的范围内随机输入。随机输入只检验数值连通性，不代表驾驶准确率。

In [ ]:
SAMPLE = ROOT / 'artifacts' / 'sample_recording_1779374497'
if (SAMPLE / 'auto_data.csv').is_file():
    images = preprocess_record(read_records([SAMPLE], limit=1)[0])
    input_kind = 'recorded_three_camera_frame'
else:
    images = np.random.default_rng(20261014).uniform(-0.5, 0.5, (1,3,3,66,200)).astype(np.float32)
    input_kind = 'synthetic_shape_check_only'
golden = Runtime(BUNDLE, mode='golden', threads=1)
expected = golden.run(images)
print(input_kind, images.shape, images.dtype)
print({name: value.tolist() for name, value in expected.items()})

## 2. 分图一致性
split-ort 的四个 FFN 仍由 CPU ORT 计算；split-numpy 用数学 erf 的软件替身。五段 PS 子图保留 attention/LN/残差/输出头。检查四输出及整数轮速差异，不能只看形状。

In [ ]:
split = Runtime(BUNDLE, mode='split-ort', threads=1)
software = Runtime(BUNDLE, mode='split-numpy', threads=1)
for label, runner in [('split-ort', split), ('split-numpy', software)]:
    actual = runner.run(images)
    errors = {name: float(np.max(np.abs(actual[name] - expected[name]))) for name in expected}
    for name in expected:
        np.testing.assert_allclose(actual[name], expected[name], rtol=0, atol=1e-3)
    mismatches = int(np.count_nonzero(np.trunc(actual['speed_raw']) != np.trunc(expected['speed_raw'])))
    print(label, {'max_abs_error': errors, 'integer_mismatches': mismatches, 'host_ms': runner.last_timings_ms})
print({spec['name']: spec['shape'] for spec in split.abi['layer_specs']})

## 3. PL 显式安全开关（默认关闭）
只有正确 bit/hwh、CMA、工具路径和板型均已核验，且没有其他任务使用PL时，才允许设为 True。Overlay 会真实配置硬件；不是只读检查。填写本次构建真实路径及实例名。不要运行原ROS车控。
**超时后不要重复执行，不要释放/退出kernel；保持进程存活，先按已验证流程复位PL/AXI master。**

In [ ]:
ALLOW_PL_DOWNLOAD = False
BITSTREAM = Path('/absolute/verified/path/ffn.bit')
IP_NAME = 'ffn_accel_0'
if ALLOW_PL_DOWNLOAD:
    if not BITSTREAM.is_file() or not BITSTREAM.with_suffix('.hwh').is_file():
        raise FileNotFoundError('需要同一次构建的同名 bit/hwh')
    from pynq import Overlay
    from ps.pynq_backend import PynqFFN
    if 'backend' in globals():
        raise RuntimeError('已有backend，请不要重复配置正在使用的PL')
    overlay = Overlay(str(BITSTREAM))
    if IP_NAME not in overlay.ip_dict:
        raise KeyError(f'请按真实IP名修改IP_NAME: {list(overlay.ip_dict)}')
    ip = getattr(overlay, IP_NAME)
    print(ip.register_map)
    backend = PynqFFN(ip, overlay.ip_dict[IP_NAME]['registers'], timeout_s=5.0)
else:
    print('PL 未加载；本次仅CPU演示。')

## 4. 先验四层真实输入，再验整个混合模型
单层使用 split-ort 最近一次运行记录的真正 FFN 输入，避免只测全零。验收顺序先4-token cross，再660-token camera。容差是首版FP32检查门槛，不是最终驾驶安全认证。

In [ ]:
if ALLOW_PL_DOWNLOAD:
    for layer_name in ['cross_0', 'cross_1', 'camera_0', 'camera_1']:
        x = split.last_ffn_inputs[layer_name]
        reference = split.last_ffn_outputs[layer_name]
        result = backend.forward(layer_name, x, split.weights[layer_name])
        np.testing.assert_allclose(result, reference, rtol=1e-4, atol=1e-4)
        print(layer_name, {'max_abs_error': float(np.max(np.abs(result-reference))), 'timing_ms': backend.last_timing})
    hybrid = Runtime(BUNDLE, mode='split-numpy', threads=1, backend=backend)
    result = hybrid.run(images)
    for name in expected:
        np.testing.assert_allclose(result[name], expected[name], rtol=0, atol=1e-3)
    print({'hybrid_ms': hybrid.last_timings_ms, 'speed_raw': result['speed_raw'].tolist()})
    print('hardware_execution=True，仅本次正确加载且通过上述断言时成立。')
else:
    print('跳过PL单层及混合模型验收，不代表通过。')

## 5. 正常释放
只在无未完成硬件访问时释放；发生超时，close 会拒绝释放。不要以重启kernel作为绕过这条保护的方法。硬件复位和异常恢复要按真实工程验证，本notebook不自动执行。

In [ ]:
if ALLOW_PL_DOWNLOAD and 'backend' in globals():
    backend.close()
    print('已正常释放FFN缓冲区。')
else:
    print('没有创建硬件缓冲区。')